# 5. United Kingdom Longitudinal Step Trends by Age Group and Sex

Author: Bram Schonfeldt

This notebook focuses on trends in average steps in the United Kingdom broken
down by different demographic groups.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import pickle as pkl
import sys
from typing import Any, Dict, List, Sequence, Tuple

from IPython.display import display
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# Use matplotlib defaults as far as possible
plt.rcdefaults()
plt.rcParams['figure.dpi'] = 200

sys.path.insert(0, os.path.abspath('../..'))
import stratify as strat

Load in the data...

In [ ]:
internal_dir = strat.get_internal_data_dir()
public_dir = strat.get_public_data_dir()
START_DATE = '2015-01-01'
END_DATE = '2026-01-01'
N_SHARDS = 100

UN_POP_MALE_PATH = (
    f'{public_dir}/WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx'
)
UN_POP_FEMALE_PATH = (
    f'{public_dir}/WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx'
)

# Load pre-computed Fitbit stratified step grid from internal_dir.
step_grid = strat.StratifySteps.from_shards(
    path_base=internal_dir,
    start_date=START_DATE,
    end_date=END_DATE,
    n_shards=N_SHARDS,
    un_pop_male_path=UN_POP_MALE_PATH,
    un_pop_female_path=UN_POP_FEMALE_PATH,
    min_days_per_user=1,
    min_age=20,
    max_age=104,
    min_steps=None,
    max_steps=172800,
    target_country=None,
    start_month=None,
    stop_month=None,
    exclude_days_of_week=None,
    override_hash='a36731bea99396ebe673f464d73ada56',
)

if step_grid.un_pop is None:
  step_grid.un_pop = step_grid.load_un_population_data()

print('Data loading complete.')

Processes the United Kingdom into 10 year age groups, and 20 year age groups.

In [ ]:
def assign_midage_10(age: float) -> int:
  """Maps individual age (20+) to 10-year midpoint bracket.

  Args:
      age: Age or 5-year stratum midpoint (20+).

  Returns:
      Midpoint integer (25 for 20-29, 35 for 30-39, ..., 75 for 70+).
  """
  if age < 70:
    return int(age) // 10 * 10 + 5
  return 75  # 75 represents 70+ bracket


def assign_age_bracket_3(midage_10: int) -> str:
  """Maps 10-year midpoint (25..75) into 20-40, 40-60, and 60+ age groups."""
  if midage_10 < 40:
    return '20-40'
  elif midage_10 < 60:
    return '40-60'
  return '60+'


AGE_10_LABEL_MAP = {
    25: '20-29',
    35: '30-39',
    45: '40-49',
    55: '50-59',
    65: '60-69',
    75: '70+',
}
AGE_10_ORDER = ['20-29', '30-39', '40-49', '50-59', '60-69', '70+']
AGE_3_ORDER = ['20-40', '40-60', '60+']

# 1. Filter Fitbit step_grid for United Kingdom, ages 20+
fb_uk_raw = step_grid.results_grid.reset_index()
fb_uk_raw = fb_uk_raw[
    (fb_uk_raw['country'] == 'United Kingdom')
    & (fb_uk_raw['midage'] >= 20)
    & (fb_uk_raw['year'] >= 2015)
    & (fb_uk_raw['year'] <= 2024)
].copy()

fb_uk_raw['midage_10'] = fb_uk_raw['midage'].apply(assign_midage_10)
fb_uk_raw['gender'] = fb_uk_raw['gender'].replace({0: 'male', 1: 'female'})

# 2. Aggregate sufficient statistics by 10-year age group (20-29 through 70+)
df_uk_10yr = (
    fb_uk_raw.groupby(['year', 'gender', 'midage_10'])[
        ['steps', 'steps_sq', 'readings']
    ]
    .sum()
    .reset_index()
)
df_uk_10yr['mean_steps_fitbit'] = df_uk_10yr['steps'] / df_uk_10yr['readings']
df_uk_10yr['var_steps_fitbit'] = np.maximum(
    df_uk_10yr['steps_sq'] / df_uk_10yr['readings']
    - df_uk_10yr['mean_steps_fitbit'] ** 2,
    0.0,
)
df_uk_10yr['std_steps_fitbit'] = np.sqrt(df_uk_10yr['var_steps_fitbit'])
df_uk_10yr = df_uk_10yr.rename(columns={'readings': 'n_fitbit'})
df_uk_10yr['se_steps_fitbit'] = df_uk_10yr['std_steps_fitbit'] / np.sqrt(
    df_uk_10yr['n_fitbit']
)
df_uk_10yr['ci95_steps_fitbit'] = 1.96 * df_uk_10yr['se_steps_fitbit']
df_uk_10yr['age_group'] = pd.Categorical(
    df_uk_10yr['midage_10'].map(AGE_10_LABEL_MAP),
    categories=AGE_10_ORDER,
    ordered=True,
)

# Merge UN population counts for the United Kingdom (ages 20+)
uk_pop_count = step_grid.un_pop[
    (step_grid.un_pop['country'] == 'United Kingdom')
    & (step_grid.un_pop['age'] >= 20)
].copy()
uk_pop_count['midage_10'] = uk_pop_count['age'].apply(assign_midage_10)
uk_pop_count['gender'] = uk_pop_count['gender'].replace(
    {0: 'male', 1: 'female'}
)
uk_pop_count = (
    uk_pop_count.groupby(['year', 'gender', 'midage_10'])['Count']
    .sum()
    .reset_index()
)
# UN WPP population counts are reported in thousands
uk_pop_count['n_un'] = uk_pop_count['Count'] * 1000.0

df_uk_10yr = df_uk_10yr.merge(
    uk_pop_count[['year', 'gender', 'midage_10', 'n_un']],
    on=['year', 'gender', 'midage_10'],
    how='left',
)

# 3. Aggregate into the 3 broad age brackets: 20-40, 40-60, and 60+
fb_uk_raw['age_bracket'] = fb_uk_raw['midage_10'].apply(assign_age_bracket_3)
df_uk_3grp = (
    fb_uk_raw.groupby(['year', 'gender', 'age_bracket'])[
        ['steps', 'steps_sq', 'readings']
    ]
    .sum()
    .reset_index()
)
df_uk_3grp['mean_steps_fitbit'] = df_uk_3grp['steps'] / df_uk_3grp['readings']
df_uk_3grp['var_steps_fitbit'] = np.maximum(
    df_uk_3grp['steps_sq'] / df_uk_3grp['readings']
    - df_uk_3grp['mean_steps_fitbit'] ** 2,
    0.0,
)
df_uk_3grp['std_steps_fitbit'] = np.sqrt(df_uk_3grp['var_steps_fitbit'])
df_uk_3grp = df_uk_3grp.rename(columns={'readings': 'n_fitbit'})
df_uk_3grp['se_steps_fitbit'] = df_uk_3grp['std_steps_fitbit'] / np.sqrt(
    df_uk_3grp['n_fitbit']
)
df_uk_3grp['ci95_steps_fitbit'] = 1.96 * df_uk_3grp['se_steps_fitbit']
df_uk_3grp['age_bracket'] = pd.Categorical(
    df_uk_3grp['age_bracket'],
    categories=AGE_3_ORDER,
    ordered=True,
)
df_uk_3grp = df_uk_3grp.sort_values(
    ['year', 'gender', 'age_bracket']
).reset_index(drop=True)

print(
    f'Processed UK 10-year age-group table: {len(df_uk_10yr)} stratum-year rows'
    f' ({df_uk_10yr["year"].min()}–{df_uk_10yr["year"].max()}).'
)
print(
    'Processed UK 3-bracket age-group table (20-40, 40-60, 60+):'
    f' {len(df_uk_3grp)} stratum-year rows.'
)
display(df_uk_3grp.head(12))

## Average steps by age group
We plot the average steps per year, broken down by age groups for male and
female participants. We use marker size to indicate population size and whiskers
to indicate uncertainty. We then fit OLS regression lines, omitting 2020-2021
to show the overall trend in steps.

In [ ]:
"""Plot UK annual average steps per year for 20-40, 40-60, and 60+ age groups by sex with OLS fits."""

plt.rcdefaults()


def plot_uk_forest_trends(
    df: pd.DataFrame,
    group_col: str,
    group_order: Sequence[str],
    err_col: str = 'ci95_steps_fitbit',
    err_label: str = '95% CI',
    stagger_width: float = 0.36,
    exclude_years: Sequence[int] = (2020, 2021),
    title: str = 'UK Average Daily Steps per Year by Age Group and Sex',
) -> Tuple[plt.Figure, pd.DataFrame]:
  """Plots forest-style staggered square-and-whisker step trends with OLS regression lines and 95% CI bands."""
  fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.5), sharey=True)
  sex_panels = [
      ('male', 'Male Participants'),
      ('female', 'Female Participants'),
  ]

  n_groups = len(group_order)
  if n_groups > 1:
    offsets = np.linspace(-stagger_width / 2.0, stagger_width / 2.0, n_groups)
  else:
    offsets = [0.0]

  # Scale square marker areas proportionally to participant count (n_fitbit)
  max_n = float(df['n_fitbit'].max())
  min_area, max_area = 25.0, 180.0

  def n_to_area(n_val: Any) -> np.ndarray:
    return min_area + (max_area - min_area) * (np.asarray(n_val) / max_n)

  prop_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
  colors = {
      grp: prop_cycle[i % len(prop_cycle)] for i, grp in enumerate(group_order)
  }

  years = sorted(df['year'].unique())
  min_yr, max_yr = float(min(years)), float(max(years))
  x_grid = np.linspace(min_yr, max_yr, 200)

  ols_records: List[Dict[str, Any]] = []

  for ax, (sex_key, panel_title) in zip(axes, sex_panels):
    # Lightly shade omitted pandemic years (2020-2021 inclusive)
    if exclude_years:
      ax.axvspan(
          min(exclude_years) - 0.45,
          max(exclude_years) + 0.45,
          color='gray',
          alpha=0.10,
          zorder=0,
      )

    sub_sex = df[df['gender'] == sex_key]
    for grp, offset in zip(group_order, offsets):
      sub_grp = sub_sex[sub_sex[group_col] == grp].sort_values('year')
      if sub_grp.empty:
        continue

      x_years = sub_grp['year'].to_numpy(dtype=float)
      x_pos = x_years + offset
      y_mean = sub_grp['mean_steps_fitbit'].to_numpy(dtype=float)
      y_err = sub_grp[err_col].to_numpy(dtype=float)
      areas = n_to_area(sub_grp['n_fitbit'].to_numpy(dtype=float))
      color = colors[grp]

      # Fit OLS regression omitting pandemic years (2020-2021 inclusive)
      fit_mask = ~sub_grp['year'].isin(exclude_years)
      x_fit = x_years[fit_mask]
      y_fit = y_mean[fit_mask]
      n_pts = len(x_fit)

      if n_pts >= 3:
        reg = stats.linregress(x_fit, y_fit)
        slope = float(reg.slope)
        intercept = float(reg.intercept)
        r2 = float(reg.rvalue**2)
        p_val = float(reg.pvalue)
        slope_se = float(reg.stderr)

        df_resid = n_pts - 2
        t_crit = float(stats.t.ppf(0.975, df=df_resid))
        y_pred_fit = slope * x_fit + intercept
        s_yx = float(np.sqrt(np.sum((y_fit - y_pred_fit) ** 2) / df_resid))
        x_bar = float(np.mean(x_fit))
        s_xx = float(np.sum((x_fit - x_bar) ** 2))

        y_hat_grid = slope * x_grid + intercept
        se_fit_grid = s_yx * np.sqrt(
            1.0 / n_pts + ((x_grid - x_bar) ** 2) / s_xx
        )
        ci_grid = t_crit * se_fit_grid

        x_grid_plot = x_grid + offset
        # 95% confidence band around the OLS regression fit
        ax.fill_between(
            x_grid_plot,
            y_hat_grid - ci_grid,
            y_hat_grid + ci_grid,
            color=color,
            alpha=0.16,
            zorder=1,
        )
        # Upper and lower 95% CI boundary lines around the fit
        ax.plot(
            x_grid_plot,
            y_hat_grid - ci_grid,
            linestyle=':',
            linewidth=0.8,
            alpha=0.55,
            color=color,
            zorder=2,
        )
        ax.plot(
            x_grid_plot,
            y_hat_grid + ci_grid,
            linestyle=':',
            linewidth=0.8,
            alpha=0.55,
            color=color,
            zorder=2,
        )
        # OLS regression line across years
        ax.plot(
            x_grid_plot,
            y_hat_grid,
            linestyle='--',
            linewidth=1.6,
            alpha=0.90,
            color=color,
            zorder=2,
        )

        slope_ci_low = slope - t_crit * slope_se
        slope_ci_high = slope + t_crit * slope_se
        y_2015 = slope * 2015.0 + intercept
        eq_str = f'Steps = {slope:+.2f} * Year {intercept:+.1f}'
        eq_2015_str = f'Steps = {slope:+.2f} * (Year - 2015) + {y_2015:.1f}'
        p_str = '< 0.0001' if p_val < 0.0001 else f'{p_val:.4f}'
        ols_records.append({
            'Sex': panel_title.replace(' Participants', ''),
            'Age Group': str(grp),
            'OLS Equation (Year)': eq_str,
            'OLS Equation (Centered 2015)': eq_2015_str,
            'Slope (steps/yr)': round(slope, 2),
            'Slope 95% CI': f'[{slope_ci_low:+.2f}, {slope_ci_high:+.2f}]',
            'R^2': round(r2, 4),
            'p-value': p_str,
            'Fit RMSE (steps)': round(s_yx, 1),
            'Years Fit (N)': n_pts,
        })

      # Square markers sized by participant count (n_fitbit)
      ax.scatter(
          x_pos,
          y_mean,
          s=areas,
          marker='s',
          color=color,
          edgecolors='none',
          zorder=3,
      )

      # Forest-plot style whiskers (95% CI around the mean)
      ax.errorbar(
          x_pos,
          y_mean,
          yerr=y_err,
          fmt='none',
          ecolor=color,
          elinewidth=1.2,
          capsize=3,
          zorder=4,
      )

    ax.set_title(panel_title)
    ax.set_xlabel('Year')
    ax.set_xticks(years)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, linestyle='--', alpha=0.35)

  axes[0].set_ylabel(f'Mean Daily Steps (with {err_label} error bars)')

  # Build combined legend for Age Groups, OLS fit band, and Participant Count (marker size)
  legend_handles = []
  for grp in group_order:
    legend_handles.append(
        mlines.Line2D(
            [0],
            [0],
            color=colors[grp],
            marker='s',
            linestyle='--',
            linewidth=1.5,
            markersize=6,
            label=f'Age {grp} (OLS & {err_label})',
        )
    )

  legend_handles.append(
      mpatches.Patch(
          facecolor='gray',
          edgecolor='gray',
          linestyle=':',
          alpha=0.25,
          label='OLS 95% CI Band',
      )
  )
  if exclude_years:
    legend_handles.append(
        mpatches.Patch(
            facecolor='gray',
            alpha=0.12,
            label=(
                f'Omitted from Fit ({min(exclude_years)}–{max(exclude_years)})'
            ),
        )
    )

  # Reference participant sizes in the legend
  ref_counts = [
      int(round(df['n_fitbit'].min(), -2)),
      int(round(df['n_fitbit'].median(), -3)),
      int(round(df['n_fitbit'].max(), -3)),
  ]
  for ref_n in sorted(set(ref_counts)):
    if ref_n <= 0:
      continue
    ref_ms = float(np.sqrt(n_to_area(ref_n)))
    legend_handles.append(
        mlines.Line2D(
            [0],
            [0],
            color='gray',
            marker='s',
            linestyle='None',
            markersize=ref_ms,
            alpha=0.7,
            label=f'N = {ref_n:,}',
        )
    )

  axes[1].legend(
      handles=legend_handles,
      title='Age Group, OLS Fit & N',
      loc='upper left',
      bbox_to_anchor=(1.02, 1.0),
      frameon=True,
  )

  fig.suptitle(title, y=1.02)
  fig.tight_layout()
  ols_df = pd.DataFrame(ols_records)
  return fig, ols_df


fig_3grp, ols_3grp_df = plot_uk_forest_trends(
    df=df_uk_3grp,
    group_col='age_bracket',
    group_order=AGE_3_ORDER,
    err_col='ci95_steps_fitbit',
    err_label='95% CI',
    stagger_width=0.36,
    exclude_years=(2020, 2021),
    title=(
        'UK Average Daily Steps per Year by Age Bracket (20-40, 40-60, 60+) and'
        ' Sex'
    ),
)
plt.show()
display(ols_3grp_df)

In [ ]:
"""Plot UK annual average steps per year across all 10-year age groups (20-29 to 70+) with OLS fits."""

fig_10yr, ols_10yr_df = plot_uk_forest_trends(
    df=df_uk_10yr,
    group_col='age_group',
    group_order=AGE_10_ORDER,
    err_col='ci95_steps_fitbit',
    err_label='95% CI',
    stagger_width=0.55,
    exclude_years=(2020, 2021),
    title=(
        'UK Average Daily Steps per Year by 10-Year Age Group (Ages 20+)'
        ' and Sex'
    ),
)
plt.show()
display(ols_10yr_df)

## Relationship between average steps and age by pandemic group

We plot the average steps against 5-year age groups. We generally expect to see
a decrease in average steps with age. However, it is interesting to contrast
the step-age curve between the pre-pandemic (2018-19), pandemic (2020-21) and
post-pandemic cohort (2022-23), since it highlights which demographic groups did
more or fewer steps during this period.

In [ ]:
"""Plot UK average daily steps vs.

age (x-axis) broken down by Pre-Pandemic (2018-2019), Pandemic (2020-2021), and
Post-Pandemic (2022-2023) year groups.
"""


def assign_pandemic_period(year: int) -> str:
  """Maps calendar year (2018..2023) into Pre-Pandemic, Pandemic, and Post-Pandemic periods."""
  if year in (2018, 2019):
    return 'Pre-Pandemic (2018–2019)'
  elif year in (2020, 2021):
    return 'Pandemic (2020–2021)'
  elif year in (2022, 2023):
    return 'Post-Pandemic (2022–2023)'
  return 'Other'


PERIOD_ORDER = [
    'Pre-Pandemic (2018–2019)',
    'Pandemic (2020–2021)',
    'Post-Pandemic (2022–2023)',
]

# Aggregate UK 5-year age strata (ages 20-79) by pandemic period and sex
fb_uk_period_raw = fb_uk_raw[
    (fb_uk_raw['midage'] >= 20)
    & (fb_uk_raw['midage'] < 80)
    & (fb_uk_raw['year'] >= 2018)
    & (fb_uk_raw['year'] <= 2023)
].copy()
fb_uk_period_raw['period'] = fb_uk_period_raw['year'].apply(
    assign_pandemic_period
)

df_uk_age_period = (
    fb_uk_period_raw.groupby(['period', 'gender', 'midage'], observed=True)[
        ['steps', 'steps_sq', 'readings']
    ]
    .sum()
    .reset_index()
)
df_uk_age_period['mean_steps_fitbit'] = (
    df_uk_age_period['steps'] / df_uk_age_period['readings']
)
df_uk_age_period['var_steps_fitbit'] = np.maximum(
    df_uk_age_period['steps_sq'] / df_uk_age_period['readings']
    - df_uk_age_period['mean_steps_fitbit'] ** 2,
    0.0,
)
df_uk_age_period['std_steps_fitbit'] = np.sqrt(
    df_uk_age_period['var_steps_fitbit']
)
df_uk_age_period = df_uk_age_period.rename(columns={'readings': 'n_fitbit'})
df_uk_age_period['se_steps_fitbit'] = df_uk_age_period[
    'std_steps_fitbit'
] / np.sqrt(df_uk_age_period['n_fitbit'])
df_uk_age_period['ci95_steps_fitbit'] = (
    1.96 * df_uk_age_period['se_steps_fitbit']
)
df_uk_age_period['period'] = pd.Categorical(
    df_uk_age_period['period'],
    categories=PERIOD_ORDER,
    ordered=True,
)
df_uk_age_period = df_uk_age_period.sort_values(
    ['period', 'gender', 'midage']
).reset_index(drop=True)


def plot_uk_steps_by_age_and_period(
    df: pd.DataFrame,
    period_order: Sequence[str] = PERIOD_ORDER,
    err_col: str = 'ci95_steps_fitbit',
    err_label: str = '95% CI',
    stagger_width: float = 1.3,
    title: str = (
        'UK Average Daily Steps by Age Across Pre-Pandemic (2018–2019),'
        ' Pandemic (2020–2021), and Post-Pandemic (2022–2023) Periods'
    ),
) -> plt.Figure:
  """Plots steps vs.

  age (x-axis) broken down by pandemic year groups for Male and Female
  participants.
  """
  fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.5), sharey=True)
  sex_panels = [
      ('male', 'Male Participants'),
      ('female', 'Female Participants'),
  ]

  n_groups = len(period_order)
  if n_groups > 1:
    offsets = np.linspace(-stagger_width / 2.0, stagger_width / 2.0, n_groups)
  else:
    offsets = [0.0]

  max_n = float(df['n_fitbit'].max())
  min_area, max_area = 25.0, 180.0

  def n_to_area(n_val: Any) -> np.ndarray:
    return min_area + (max_area - min_area) * (np.asarray(n_val) / max_n)

  prop_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
  colors = {
      prd: prop_cycle[i % len(prop_cycle)] for i, prd in enumerate(period_order)
  }

  midages = sorted(df['midage'].unique())
  age_tick_labels = [f'{int(m - 2.5)}–{int(m + 1.5)}' for m in midages]

  for ax, (sex_key, panel_title) in zip(axes, sex_panels):
    sub_sex = df[df['gender'] == sex_key]
    for prd, offset in zip(period_order, offsets):
      sub_prd = sub_sex[sub_sex['period'] == prd].sort_values('midage')
      if sub_prd.empty:
        continue

      x_ages = sub_prd['midage'].to_numpy(dtype=float)
      x_pos = x_ages + offset
      y_mean = sub_prd['mean_steps_fitbit'].to_numpy(dtype=float)
      y_err = sub_prd[err_col].to_numpy(dtype=float)
      areas = n_to_area(sub_prd['n_fitbit'].to_numpy(dtype=float))
      color = colors[prd]

      # Connecting line across 5-year age strata
      ax.plot(
          x_pos,
          y_mean,
          linestyle='-',
          linewidth=1.4,
          alpha=0.65,
          color=color,
          zorder=2,
      )

      # Square markers sized by participant count (n_fitbit)
      ax.scatter(
          x_pos,
          y_mean,
          s=areas,
          marker='s',
          color=color,
          edgecolors='none',
          zorder=3,
      )

      # Forest-plot style whiskers (95% CI around the mean)
      ax.errorbar(
          x_pos,
          y_mean,
          yerr=y_err,
          fmt='none',
          ecolor=color,
          elinewidth=1.2,
          capsize=3,
          zorder=4,
      )

    ax.set_title(panel_title)
    ax.set_xlabel('Age Group (5-Year Strata, Ages 20–79)')
    ax.set_xticks(midages)
    ax.set_xticklabels(age_tick_labels, rotation=45)
    ax.grid(True, linestyle='--', alpha=0.35)

  axes[0].set_ylabel(f'Mean Daily Steps (with {err_label} error bars)')

  legend_handles = []
  for prd in period_order:
    legend_handles.append(
        mlines.Line2D(
            [0],
            [0],
            color=colors[prd],
            marker='s',
            linestyle='-',
            linewidth=1.4,
            markersize=6,
            label=f'{prd}',
        )
    )

  ref_counts = [
      int(round(df['n_fitbit'].min(), -2)),
      int(round(df['n_fitbit'].median(), -3)),
      int(round(df['n_fitbit'].max(), -3)),
  ]
  for ref_n in sorted(set(ref_counts)):
    if ref_n <= 0:
      continue
    ref_ms = float(np.sqrt(n_to_area(ref_n)))
    legend_handles.append(
        mlines.Line2D(
            [0],
            [0],
            color='gray',
            marker='s',
            linestyle='None',
            markersize=ref_ms,
            alpha=0.7,
            label=f'N = {ref_n:,}',
        )
    )

  axes[1].legend(
      handles=legend_handles,
      title='Year Group & Participants',
      loc='upper left',
      bbox_to_anchor=(1.02, 1.0),
      frameon=True,
  )

  fig.suptitle(title, y=1.02)
  fig.tight_layout()
  return fig


fig_age_period = plot_uk_steps_by_age_and_period(df_uk_age_period)
plt.show()

# *FIN*